# PlantSeg — YOLOv8 Training Pipeline

End-to-end training of a YOLOv8 plant-disease detector on the
[**PlantSeg**](https://arxiv.org/abs/2409.04038) in-the-wild dataset
(11,400+ images, 115 disease classes).

This notebook is designed to run **top to bottom**. It:

1. Converts the COCO instance annotations to YOLO label format (clamping the
   out-of-bounds boxes the raw dataset ships with).
2. Trains a YOLOv8 detector (size configurable via `MODEL_SIZE`).
3. Evaluates it and reports COCO metrics (mAP50 / mAP50-95).
4. Runs inference on a few test images for a visual sanity check.
5. Copies `best.pt` to the Django web app so it is picked up automatically.

> Reproduce the reported model: `MODEL_SIZE=s IMGSZ=640` (YOLOv8s, ~26 FPS on
> a mid-range laptop CPU at `imgsz=416`, early-stopped at epoch 95).

The optional **Faster R-CNN baseline** lives in the appendix at the end.

In [ ]:
import os
import json
import shutil
from collections import defaultdict
from pathlib import Path

from ultralytics import YOLO

# ─── Configuration ───────────────────────────────────────────────────────
# The PlantSeg dataset (images + COCO masks) must be staged here:
DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "plantsegv3"))

# Model to train: n / s / m / l / x
MODEL_SIZE = os.environ.get("MODEL_SIZE", "s")

# Training hyper-parameters (overridable via env vars)
IMGSZ      = int(os.environ.get("IMGSZ", 640))
EPOCHS     = int(os.environ.get("EPOCHS", 120))
BATCH      = int(os.environ.get("BATCH", 16))
PATIENCE   = int(os.environ.get("PATIENCE", 30))
SAVE_PERIOD = int(os.environ.get("SAVE_PERIOD", 10))

PROJECT_DIR   = Path("runs/detect")
EXPERIMENT    = f"yolov8{MODEL_SIZE}_imgsz{IMGSZ}"

# Where the trained weights go so the Django app picks them up automatically.
APP_WEIGHTS = Path("plant_disease") / "best.pt"

assert DATASET_ROOT.exists(), f"Dataset not found at {DATASET_ROOT}. Place PlantSeg here (see README)."
print(f"Dataset root: {DATASET_ROOT.resolve()}")

## 1. Prepare the dataset (COCO → YOLO)

PlantSeg ships COCO-format instance annotations
(`masks/annotation_*.json`). YOLO expects one `<image>.txt` label file per
image with normalized `cls cx cy w h` lines.

Two things are handled here that matter for training quality:

- **Out-of-bounds boxes** — many raw annotations extend past the image edge.
  They are clamped to the image and degenerate boxes are dropped.
- **Non-contiguous category ids** — COCO ids are re-mapped to dense YOLO
  class indices so `model.names` matches the 115 disease classes.

Run this after staging the dataset at `DATASET_ROOT`:
`images/{train,val,test}/` and `masks/annotation_{train,val,test}.json`.

In [ ]:
def convert_coco_to_yolo(coco_json, labels_dir, min_box_size=1.0):
    """Convert one COCO annotation file to YOLO labels, clamping boxes."""
    coco = json.loads(Path(coco_json).read_text())

    cats = sorted(coco["categories"], key=lambda c: c["id"])
    id2index = {c["id"]: i for i, c in enumerate(cats)}
    classes = [c["name"] for c in cats]

    images = {img["id"]: img for img in coco["images"]}
    anns_by_image = defaultdict(list)
    for ann in coco["annotations"]:
        anns_by_image[ann["image_id"]].append(ann)

    labels_dir = Path(labels_dir)
    labels_dir.mkdir(parents=True, exist_ok=True)

    n_imgs = n_degenerate = 0
    for img_id, img in images.items():
        w, h = img["width"], img["height"]
        lines = []
        for ann in anns_by_image.get(img_id, []):
            if ann.get("iscrowd", 0):
                continue
            x, y, bw, bh = ann["bbox"]
            x1 = max(0.0, min(float(x), w))
            y1 = max(0.0, min(float(y), h))
            x2 = max(0.0, min(float(x) + bw, w))
            y2 = max(0.0, min(float(y) + bh, h))
            if (x2 - x1) < min_box_size or (y2 - y1) < min_box_size:
                n_degenerate += 1
                continue
            cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
            lines.append(f"{id2index[ann['category_id']]} {cx:.6f} {cy:.6f} {(x2 - x1) / w:.6f} {(y2 - y1) / h:.6f}")
        (labels_dir / f"{Path(img['file_name']).stem}.txt").write_text("\n".join(lines) + "\n")
        n_imgs += 1

    (Path(labels_dir).parent / "classes.txt").write_text("\n".join(classes) + "\n")
    print(f"{Path(coco_json).name}: {n_imgs} images -> labels, {n_degenerate} degenerate boxes dropped")
    return classes

classes = None
for split in ("train", "val", "test"):
    coco_json = DATASET_ROOT / "masks" / f"annotation_{split}.json"
    if not coco_json.exists():
        print(f"Skip {split}: {coco_json} not found")
        continue
    labels_dir = DATASET_ROOT / "labels" / split
    classes = convert_coco_to_yolo(coco_json, labels_dir)

assert classes, "No annotations found — check the dataset staging (README)."
print(f"{len(classes)} disease classes")

In [ ]:
# Generate data.yaml for Ultralytics (uses the labels created above).
nc = len(classes)
data_yaml = DATASET_ROOT / "data.yaml"
data_yaml.write_text(
    f"path: {DATASET_ROOT.resolve()}\n"
    f"train: images/train\n"
    f"val: images/val\n"
    f"test: images/test\n"
    f"nc: {nc}\n"
    f"names: {classes!r}\n"
)
print(data_yaml.read_text())

## 2. Train YOLOv8

Ultralytics YOLOv8 is trained from the COCO-pretrained checkpoint of the
requested size. Training runs on GPU when available (e.g. CETUS HPC / AWS
SageMaker) and falls back to CPU otherwise — `epochs` / `patience` are tuned
so early stopping usually kicks in well before `EPOCHS`.

In [ ]:
model = YOLO(f"yolov8{MODEL_SIZE}.pt")
model.train(
    data=str(data_yaml),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    save_period=SAVE_PERIOD,
    project=str(PROJECT_DIR),
    name=EXPERIMENT,
    device=0 if __import__("torch").cuda.is_available() else "cpu",
)
print("Training done — best weights at", PROJECT_DIR / EXPERIMENT / "weights" / "best.pt")

## 3. Evaluate on the validation split

Reports the standard COCO detection metrics used in the project report
(validation set: 1,247 images / 8,926 instances).

In [ ]:
metrics = model.val(data=str(data_yaml), split="val")
box = metrics.box
print(f"mAP50      : {box.map50:.3f}")
print(f"mAP50-95   : {box.map:.3f}")
print(f"Precision  : {box.p:.3f}")
print(f"Recall     : {box.r:.3f}")

## 4. Visual sanity check on test images

Runs the trained model on a handful of held-out test images and saves the
annotated results under `runs/detect/{EXPERIMENT}/sample_inference/`.

In [ ]:
import cv2

test_images = sorted((DATASET_ROOT / "images" / "test").glob("*"))[:8]
out_dir = PROJECT_DIR / EXPERIMENT / "sample_inference"
out_dir.mkdir(parents=True, exist_ok=True)

for path in test_images:
    results = model.predict(str(path), imgsz=IMGSZ, conf=0.10, verbose=False)
    annotated = results[0].plot()
    cv2.imwrite(str(out_dir / path.name), annotated)

print(f"Saved {len(test_images)} annotated images to {out_dir}")

## 5. Export weights for the Django web app

Copies the trained `best.pt` to `plant_disease/best.pt` — the web app loads
this path automatically and falls back to a pretrained model only when it is
missing.

In [ ]:
best = PROJECT_DIR / EXPERIMENT / "weights" / "best.pt"
assert best.exists(), f"No best.pt found at {best}"
APP_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(best, APP_WEIGHTS)
print(f"Copied {best} -> {APP_WEIGHTS}")

---

## Appendix — Faster R-CNN baseline (optional)

For comparison, a Faster R-CNN (ResNet-50-FPN) baseline implemented with
`torchvision` on the raw COCO annotations. This was trained as an ablation;
YOLOv8 was used for the shipped model because it runs real-time on CPU.

Set `TRAIN_BASELINE = True` below to run it. Expect many hours on GPU for
meaningful results — it is provided as a self-contained reference.

In [ ]:
import os
import time
from pathlib import Path

import torch
import torchvision.transforms as T
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader, Dataset
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from PIL import Image

TRAIN_BASELINE = False  # flip to True to run the baseline

In [ ]:
class CocoDetectionDataset(Dataset):
    """Minimal COCO detection dataset returning (image, target) pairs."""
    def __init__(self, image_dir, annotation_path):
        self.image_dir = Path(image_dir)
        self.coco = COCO(annotation_path)
        self.ids = sorted(self.coco.imgs.keys())

    def __getitem__(self, index):
        img_id = self.ids[index]
        anns = self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id))
        path = self.coco.loadImgs(img_id)[0]["file_name"]
        img = Image.open(self.image_dir / path).convert("RGB")

        boxes, labels = [], []
        for ann in anns:
            if ann.get("iscrowd", 0) or not ann.get("bbox"):
                continue
            x, y, w, h = ann["bbox"]
            if w > 0 and h > 0:
                boxes.append([x, y, x + w, y + h])
                labels.append(ann["category_id"])

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels, "image_id": torch.tensor(img_id)}
        return T.ToTensor()(img), target

    def __len__(self):
        return len(self.ids)


if TRAIN_BASELINE:
    train_ds = CocoDetectionDataset(DATASET_ROOT / "images" / "train", DATASET_ROOT / "masks" / "annotation_train.json")
    val_ds = CocoDetectionDataset(DATASET_ROOT / "images" / "val", DATASET_ROOT / "masks" / "annotation_val.json")
    train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, collate_fn=lambda x: tuple(zip(*x)))
    val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
    print(f"train: {len(train_ds)} images | val: {len(val_ds)} images")

In [ ]:
def train_baseline():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, nc + 1)  # +1 background
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    num_epochs = 40
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        for images, targets in train_loader:
            images = [im.to(device) for im in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()
        print(f"Epoch {epoch + 1}/{num_epochs} | train loss: {epoch_loss / len(train_loader):.4f}")

    torch.save({"model_state_dict": model.state_dict()}, "faster_rcnn_baseline.pth")
    return model


if TRAIN_BASELINE:
    baseline = train_baseline()

In [ ]:
def evaluate_baseline(model, loader, device, annotation_path, iou_type="bbox"):
    """COCO evaluation for a torchvision detector."""
    model.eval()
    coco_gt = COCO(annotation_path)
    results = []
    with torch.no_grad():
        for images, targets in loader:
            images = [im.to(device) for im in images]
            outputs = model(images)
            for output, target in zip(outputs, targets):
                image_id = target["image_id"].item()
                for box, score, label in zip(
                    output["boxes"].cpu().numpy(),
                    output["scores"].cpu().numpy(),
                    output["labels"].cpu().numpy(),
                ):
                    x1, y1, x2, y2 = box
                    results.append({
                        "image_id": int(image_id),
                        "category_id": int(label),
                        "bbox": [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                        "score": float(score),
                    })

    det_json = "baseline_detections.json"
    with open(det_json, "w") as f:
        json.dump(results, f)

    coco_dt = coco_gt.loadRes(det_json)
    evalr = COCOeval(coco_gt, coco_dt, iouType=iou_type)
    evalr.evaluate()
    evalr.accumulate()
    evalr.summarize()
    return evalr.stats


if TRAIN_BASELINE:
    test_ds = CocoDetectionDataset(DATASET_ROOT / "images" / "test", DATASET_ROOT / "masks" / "annotation_test.json")
    test_loader = DataLoader(test_ds, batch_size=4, shuffle=False, collate_fn=lambda x: tuple(zip(*x)))
    stats = evaluate_baseline(baseline, test_loader, device,
                              DATASET_ROOT / "masks" / "annotation_test.json")
    print(f"COCO mAP stats: {stats}")